## 一次请求进入 FastAPI 后，究竟经过哪些环节，最后才变成响应？
```
客户端请求
    ↓
Uvicorn
    ↓
Middleware 请求阶段
    ↓
路由匹配
    ↓
依赖解析与参数校验
    ↓
路由函数
    ↓
响应模型校验与序列化
    ↓
Middleware 响应阶段
    ↓
Uvicorn
    ↓
客户端响应
```

### Uvicorn 接收网络请求

1. 接收 TCP 连接。
2. 解析 HTTP 数据。
3. 构造 ASGI scope。
4. 通过 ASGI 接口调用 FastAPI。

### 请求进入 Middleware
Middleware 是包裹整个应用的通用处理层。

In [ ]:
@app.middleware("http")
async def log_request(request, call_next):
    print("请求进入")

    response = await call_next(request)

    print("响应离开")
    return response

执行到`await call_next(request)`时请求会继续进入后面的路由、依赖和业务逻辑。

内部处理完成后，响应会返回到 Middleware，继续执行

因此 Middleware 是一个“洋葱模型“

Middleware 常用于：
- 请求日志
- 请求耗时
- Request ID
- CORS
- 全局认证
- 链路追踪
- 响应头处理

### 路由匹配
FastAPI 根据请求方法和路径寻找对应的路由

### 解析依赖关系
它会：
- 检查路由函数声明了哪些依赖。
- 执行相应依赖。
- 获得依赖返回值。
- 注入路由函数参数。

依赖还可以拥有自己的依赖，从而形成依赖树。后面会专门学习 Depends。

In [ ]:
from typing import Annotated
from fastapi import Depends


async def get_current_user():
    return {"id": 1, "username": "Alice"}


@app.get("/profile")
async def get_profile(
    current_user: Annotated[dict, Depends(get_current_user)],
):
    return current_user

# 不需要手动调用get_current_user，FastAPI会自动执行

### 提取并校验参数
FastAPI 根据函数签名确定数据来源：
会分别处理:
- user_id  ← Path
- notify   ← Query
- payload  ← Body

然后使用类型声明和 Pydantic 完成解析与校验。

请求校验失败时，路由函数不会执行。

### 执行路由函数
只有路由匹配、依赖处理和参数校验通过后，路由函数才会执行

路由函数一般负责：
- 接收已经校验的数据
- 调用 Service
- 返回业务结果
- 必要时抛出 HTTP 或业务异常

路由函数不适合直接堆积大量 SQL、缓存和复杂业务逻辑。

推荐：Route → Service → Repository

### 响应模型处理

FastAPI 会：
- 根据响应模型检查返回数据。
- 过滤响应模型中没有的字段。
- 将结果序列化成 JSON 响应。

### 异常处理
路由中可能抛出：
```
raise HTTPException(
    status_code=404,
    detail="User not found",
)
```
正常的响应模型流程会被中断，异常交给对应的异常处理器。

异常处理器负责把异常转换成 HTTP 响应

### 响应经过 Middleware 返回
内部生成响应之后，会回到 Middleware

```
@app.middleware("http")
async def add_request_id(request, call_next):
    response = await call_next(request)
    response.headers["X-Request-ID"] = "abc-123"
    return response
```

Middleware 可以在响应离开前：
- 添加响应头
- 记录耗时
- 写访问日志
- 记录状态码
- 执行通用清理

### 面试表达
> 一次 FastAPI 请求由 Uvicorn 接收并转换成 ASGI 消息，进入 Middleware 后完成路由匹配、依赖解析和参数校验。校验通过才执行路由函数，返回值再经过响应模型校验和序列化，最后沿 Middleware 返回。如果中间抛出异常，则由对应异常处理器转换成 HTTP 响应。